# Model Results

Fits and evalutes the Bayesian model.

## Imports

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import torch

In [2]:
# Find root directory of project
root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "src" / "config.py").exists())
sys.path.insert(0, str(root))

In [3]:
from src.config import PROCESSED_DATA_DIR, RANDOM_SEED, TABLES_DIR, TRAIN_SEASONS, TEST_SEASONS
from src.model.model import BayesianPoissonModel, BayesianPoissonModelConfig, SVIConfig

/Users/astonchan/Desktop/Academics Folder/UC Irvine/Senior Year/Spring Quarter/COMPSCI 179 – Algorithms for Probabilistic and Deterministic Graphical Models/Project/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Evaluation

### Load Training and Testing Data

In [4]:
df = pd.read_csv(PROCESSED_DATA_DIR / "epl_matches.csv", parse_dates = ["Date"])

train_df = df[df["Season"].isin(TRAIN_SEASONS)].reset_index(drop = True)
test_df = df[df["Season"].isin(TEST_SEASONS)].reset_index(drop = True)

print(f"Train: {len(train_df)} matches")
print(f"Test: {len(test_df)} matches")

Train: 3040 matches
Test: 760 matches


### Ablated Bayesian Model (No Home-Team Advantage)

In [5]:
no_home_team_bayesian_model = BayesianPoissonModel(
    config = BayesianPoissonModelConfig(
        seed = RANDOM_SEED,
        device = (torch.device("cuda")
                  if torch.cuda.is_available()
                  else torch.device("cpu")),
        svi = SVIConfig(),
        ablate_attack = False,
        ablate_defense = False,
        ablate_home_team_advantage = True,
        num_posterior_samples_per_match = 100,
    )
).fit(train_df)
no_home_team_bayesian_model_preds = no_home_team_bayesian_model.predict(test_df)

Step 100: Loss = 9356.797
Step 200: Loss = 9234.684
Step 300: Loss = 9223.475
Step 400: Loss = 9193.177
Step 500: Loss = 9196.211
Step 600: Loss = 9187.513
Step 700: Loss = 9185.984
Step 800: Loss = 9184.013
Step 900: Loss = 9187.046
Step 1000: Loss = 9187.117
Step 1100: Loss = 9185.099
Step 1200: Loss = 9184.590
Step 1300: Loss = 9187.083
Step 1400: Loss = 9185.090
Step 1500: Loss = 9185.576
Step 1600: Loss = 9186.496
Step 1700: Loss = 9185.148
Step 1800: Loss = 9186.132
Step 1900: Loss = 9186.065
Step 2000: Loss = 9185.010
Step 2100: Loss = 9185.597
Step 2200: Loss = 9186.853
Step 2300: Loss = 9183.858
Step 2400: Loss = 9185.055
Step 2500: Loss = 9186.768
Step 2600: Loss = 9185.555
Step 2700: Loss = 9185.475
Step 2800: Loss = 9184.502
Step 2900: Loss = 9185.133
Step 3000: Loss = 9187.319
Step 3100: Loss = 9184.675
Step 3200: Loss = 9186.817
Step 3300: Loss = 9185.835
Step 3400: Loss = 9187.026
Step 3500: Loss = 9184.159
Step 3600: Loss = 9181.974
Step 3700: Loss = 9186.821
Step 3800:

In [6]:
no_home_team_bayesian_model_preds[["HomeTeam", "AwayTeam", "Result", "ProbHomeWin", "ProbDraw", "ProbAwayWin"]].head()

,HomeTeam,AwayTeam,Result,ProbHomeWin,ProbDraw,ProbAwayWin
0,Man United,Fulham,H,0.597963,0.223492,0.178545
1,West Ham,Aston Villa,A,0.351086,0.249856,0.399058
2,Nott'm Forest,Bournemouth,D,0.468140,0.236412,0.295448
3,Newcastle,Southampton,H,0.525708,0.241160,0.233132
4,Everton,Brighton,A,0.437855,0.272538,0.289608


In [7]:
pred_labels = no_home_team_bayesian_model_preds[["ProbHomeWin",
                                    "ProbDraw",
                                    "ProbAwayWin"]].idxmax(axis = 1).map({"ProbHomeWin": "H",
                                                                          "ProbDraw": "D",
                                                                          "ProbAwayWin": "A"})
accuracy = (pred_labels == no_home_team_bayesian_model_preds["Result"]).mean()
print(f"(No Home-Team Advantage) Bayesian Model Accuracy: {accuracy:.3f}")

(No Home-Team Advantage) Bayesian Model Accuracy: 0.455


### Ablated Bayesian Model (No Team-Based Parameters)

In [8]:
no_team_based_bayesian_model = BayesianPoissonModel(
    config = BayesianPoissonModelConfig(
        seed = RANDOM_SEED,
        device = (torch.device("cuda")
                  if torch.cuda.is_available()
                  else torch.device("cpu")),
        svi = SVIConfig(),
        ablate_attack = True,
        ablate_defense = True,
        ablate_home_team_advantage = False,
        num_posterior_samples_per_match = 100,
    )
).fit(train_df)
no_team_based_bayesian_model_preds = no_team_based_bayesian_model.predict(test_df)

Step 100: Loss = 9466.456
Step 200: Loss = 9436.298
Step 300: Loss = 9432.815
Step 400: Loss = 9423.288
Step 500: Loss = 9429.377
Step 600: Loss = 9423.353
Step 700: Loss = 9425.218
Step 800: Loss = 9425.130
Step 900: Loss = 9422.734
Step 1000: Loss = 9422.873
Step 1100: Loss = 9421.574
Step 1200: Loss = 9422.248
Step 1300: Loss = 9422.557
Step 1400: Loss = 9421.117
Step 1500: Loss = 9422.178
Step 1600: Loss = 9421.483
Step 1700: Loss = 9422.046
Step 1800: Loss = 9421.718
Step 1900: Loss = 9421.577
Step 2000: Loss = 9421.100
Step 2100: Loss = 9421.846
Step 2200: Loss = 9421.614
Step 2300: Loss = 9421.369
Step 2400: Loss = 9421.635
Step 2500: Loss = 9421.587
Step 2600: Loss = 9421.553
Step 2700: Loss = 9422.047
Step 2800: Loss = 9421.762
Step 2900: Loss = 9422.857
Step 3000: Loss = 9421.884
Step 3100: Loss = 9421.747
Step 3200: Loss = 9422.105
Step 3300: Loss = 9421.950
Step 3400: Loss = 9421.609
Step 3500: Loss = 9421.603
Step 3600: Loss = 9421.851
Step 3700: Loss = 9421.757
Step 3800:

In [9]:
no_team_based_bayesian_model_preds[["HomeTeam", "AwayTeam", "Result", "ProbHomeWin", "ProbDraw", "ProbAwayWin"]].head()

,HomeTeam,AwayTeam,Result,ProbHomeWin,ProbDraw,ProbAwayWin
0,Man United,Fulham,H,0.442811,0.247931,0.309258
1,West Ham,Aston Villa,A,0.442298,0.248137,0.309566
2,Nott'm Forest,Bournemouth,D,0.443621,0.247527,0.308852
3,Newcastle,Southampton,H,0.441904,0.248285,0.309811
4,Everton,Brighton,A,0.443137,0.247819,0.309044


In [10]:
pred_labels = no_home_team_bayesian_model_preds[["ProbHomeWin",
                                                 "ProbDraw",
                                                 "ProbAwayWin"]].idxmax(axis = 1).map({"ProbHomeWin": "H",
                                                                                       "ProbDraw": "D",
                                                                                       "ProbAwayWin": "A"})
accuracy = (pred_labels == no_home_team_bayesian_model_preds["Result"]).mean()
print(f"(No Team-Based Parameters) Bayesian Model Accuracy: {accuracy:.3f}")

(No Team-Based Parameters) Bayesian Model Accuracy: 0.455


### Save Evaluation Results

In [11]:
model_table_output_directory = (TABLES_DIR / "model")
model_table_output_directory.mkdir(parents = True, exist_ok = True)

no_team_based_bayesian_model_preds.to_csv(model_table_output_directory / "ablation_bayesian_model_predictions.csv",
                            index = False)